# BERT for Aspect-Based Sentiment Analysis (ABSA) - 10-Fold Cross-Validation

This notebook contains the complete code for BERT-based Aspect-Based Sentiment Analysis with **10-Fold Cross-Validation** for more robust evaluation on smaller datasets.

## Contents:
1. **Imports & Configuration**
2. **Data Loading** (loader.py)
3. **Tokenizer** (Tokenizer.py)
4. **Models** (model.py)
5. **Training & Evaluation with K-Fold** (train.py)
6. **Main Pipeline with 10-Fold CV** (main.py)

## 1. Imports & Configuration

In [2]:
from __future__ import print_function
import random
import copy
import os
import warnings
from ast import literal_eval
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as utils
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import BertModel, BertForSequenceClassification, BertTokenizer
from tqdm import tqdm  # Progress bar

# Suppress tokenizer warnings about overflowing tokens
warnings.filterwarnings("ignore", message=".*overflowing tokens.*")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set random seeds for reproducibility
SEED = 1345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Model hyperparameters
fchidden = 256
hiddendim_lstm = 128
embeddim = 768
numlayers = 12

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cpu':
    print("CPU")

Using device: cuda


## 2. Data Loading Functions

Functions to load SemEval and Twitter datasets for ABSA.

In [3]:
def load_semeval(dataset, num_classes, data_path=None):
    """ Loads SemEval 14 datasets

    Arguments:
        dataset (str): Name of the datsets
        num_classes (int): Number of classes to consider
        data_path (str): Path to the dataset

    """
    label = {'negative': 0,
             'positive': 1,
             'neutral': 2,
             'conflict': 3}

    if(data_path is None):
        data_path = '../Data/'
    train_file = data_path + 'atsa-' + dataset + '/atsa_train.json'
    test_file = data_path + 'atsa-' + dataset + '/atsa_test.json'

    temp = open(train_file, 'r', encoding='latin1').read()
    train = literal_eval(temp)
    train_sentence = []
    train_aspect = []
    train_sentiment = []
    for xml in train:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        train_sentence.append(xml['sentence'])
        train_aspect.append(xml['aspect'])
        train_sentiment.append(label[xml['sentiment']])

    temp = open(test_file, 'r', encoding='latin1').read()
    test = literal_eval(temp)
    test_sentence = []
    test_aspect = []
    test_sentiment = []
    for xml in test:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        test_sentence.append(xml['sentence'])
        test_aspect.append(xml['aspect'])
        test_sentiment.append(label[xml['sentiment']])

    train_sen_len = [len(sentence.split()) for sentence in train_sentence]
    train_asp_len = [len(aspect.split()) for aspect in train_aspect]

    print("----------------------------------------")
    print("Maximum Training data Sentence Length: {}".format(
                                                       max(train_sen_len)))
    print("Maximum Training data Aspect Length: {}".format(max(train_asp_len)))
    print("----------------------------------------")

    return (train_sentence, train_aspect, train_sentiment, test_sentence,
            test_aspect, test_sentiment)

In [4]:
def load_semeval_combined(dataset, num_classes, data_path=None):
    """ Loads SemEval 14 datasets and combines train+test for K-Fold CV

    Arguments:
        dataset (str): Name of the datsets
        num_classes (int): Number of classes to consider
        data_path (str): Path to the dataset

    Returns:
        all_sentences, all_aspects, all_sentiments: Combined data
    """
    label = {'negative': 0,
             'positive': 1,
             'neutral': 2,
             'conflict': 3}

    if(data_path is None):
        data_path = '../Data/'
    train_file = data_path + 'atsa-' + dataset + '/atsa_train.json'
    test_file = data_path + 'atsa-' + dataset + '/atsa_test.json'

    all_sentences = []
    all_aspects = []
    all_sentiments = []

    # Load training data
    temp = open(train_file, 'r', encoding='latin1').read()
    train = literal_eval(temp)
    for xml in train:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        all_sentences.append(xml['sentence'])
        all_aspects.append(xml['aspect'])
        all_sentiments.append(label[xml['sentiment']])

    # Load test data
    temp = open(test_file, 'r', encoding='latin1').read()
    test = literal_eval(temp)
    for xml in test:
        if(xml['sentiment'] == 'conflict' and num_classes == 3):
            continue
        all_sentences.append(xml['sentence'])
        all_aspects.append(xml['aspect'])
        all_sentiments.append(label[xml['sentiment']])

    sen_len = [len(sentence.split()) for sentence in all_sentences]
    asp_len = [len(aspect.split()) for aspect in all_aspects]

    print("----------------------------------------")
    print(f"Total samples: {len(all_sentences)}")
    print("Maximum Sentence Length: {}".format(max(sen_len)))
    print("Maximum Aspect Length: {}".format(max(asp_len)))
    print("----------------------------------------")

    return all_sentences, all_aspects, all_sentiments

In [5]:
def get_loader(input_ids, attention_masks, token_type_ids, labels, batchsize):
    """ Converts input values into a dataloader

    Arguments:
    input_ids (Tensors): Sentences converted to input ids
    attention_masks (Tensors): Attention masks of the words in a sentence
    token_type_ids (Tensors): Token ids for pair of sentences
    labels (Tensors): Labels of the Dataset
    batchsize (int): Batch size to train

    """
    array = utils.TensorDataset(input_ids, attention_masks, token_type_ids,
                                labels)
    loader = utils.DataLoader(array, batch_size=batchsize)
    return loader

## 3. Tokenizer Functions

Functions to load pretrained BERT tokenizer and tokenize sentences.

In [6]:
def get_pretrained_tokenizer():
    """
    Loads pretrained BERT Tokenizer and returns it,

    """
    print("Downloading bert tokenizer to cache")
    print("---------------------------------------")
    bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased',
                                                   do_lower_case=True
                                                   )
    return bert_tokenizer


def tokenize_sentences(bert_tokenizer, sentences, aspects, maxlen):
    """ converts sentences into ids according to bert tokenizer

    Arguments:
    bert_tokenizer (Tokenizer): Pretrained BERT Tokenizer
    sentences (list): List of sentences
    aspects (list): List of Aspects
    maxlen (int): Maximum Length of a sentence

    """
    input_ids = []
    attention_masks = []
    token_type_ids = []

    for sentence, aspect in zip(sentences, aspects):

        encoded = bert_tokenizer.encode_plus(text=sentence,
                                             text_pair=aspect,
                                             add_special_tokens=True,
                                             max_length=maxlen,
                                             padding='max_length',
                                             truncation=True,
                                             return_attention_mask=True,
                                             return_token_type_ids=True,
                                             return_tensors='pt')

        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])
        token_type_ids.append(encoded['token_type_ids'])

    input_ids = torch.cat(input_ids, dim=0, out=None)
    attention_masks = torch.cat(attention_masks, dim=0, out=None)
    token_type_ids = torch.cat(token_type_ids, dim=0, out=None)

    return input_ids, attention_masks, token_type_ids

## 4. Model Definitions

Three BERT-based models for ABSA:
- **Bert_Base**: Using BertForSequenceClassification
- **Bert_LSTM**: BERT + LSTM
- **Bert_Attention**: BERT + Attention mechanism

In [7]:
class Bert_Base(nn.Module):
    def __init__(self, numclasses):
        super(Bert_Base, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.dropout = nn.Dropout(0.1)

        self.bert = BertForSequenceClassification.from_pretrained('bert-base-uncased',
                                                                   output_hidden_states=False,
                                                                   output_attentions=False,
                                                                   num_labels=self.numclasses)
        print("BERT Model Loaded")

    def forward(self, inp_ids, att_mask, token_ids, labels):
        outputs = self.bert(input_ids=inp_ids, attention_mask=att_mask,
                            token_type_ids=token_ids, labels=labels)
        return outputs.loss, outputs.logits

In [8]:
class Bert_LSTM(nn.Module):
    def __init__(self, numclasses):
        super(Bert_LSTM, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.numlayers = numlayers
        self.hiddendim_lstm = hiddendim_lstm
        self.dropout = nn.Dropout(0.1)

        self.bert = BertModel.from_pretrained('bert-base-uncased',
                                              output_hidden_states=True,
                                              output_attentions=False)
        print("BERT Model Loaded")
        self.lstm = nn.LSTM(self.embeddim, self.hiddendim_lstm, batch_first=True)
        self.fc = nn.Linear(self.hiddendim_lstm, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(input_ids=inp_ids,
                            attention_mask=att_mask,
                            token_type_ids=token_ids)
        hidden_states = outputs.hidden_states

        hidden_states = torch.stack([hidden_states[layer_i][:, 0].squeeze()
                                     for layer_i in range(0, self.numlayers)], dim=-1)
        hidden_states = hidden_states.view(-1, self.numlayers, self.embeddim)
        out, _ = self.lstm(hidden_states, None)
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

In [9]:
class Bert_Attention(nn.Module):
    def __init__(self, numclasses, device=None):
        super(Bert_Attention, self).__init__()
        self.numclasses = numclasses
        self.embeddim = embeddim
        self.numlayers = numlayers
        self.fchidden = fchidden
        self.dropout = nn.Dropout(0.1)

        self.bert = BertModel.from_pretrained('bert-base-uncased',
                                              output_hidden_states=True,
                                              output_attentions=False)
        print("BERT Model Loaded")

        # Initialize attention parameters - don't use .to(device) here
        # Let PyTorch handle device placement when model.to(device) is called
        q_t = np.random.normal(loc=0.0, scale=0.1, size=(1, self.embeddim))
        self.q = nn.Parameter(torch.from_numpy(q_t).float())
        
        w_ht = np.random.normal(loc=0.0, scale=0.1, size=(self.embeddim, self.fchidden))
        self.w_h = nn.Parameter(torch.from_numpy(w_ht).float())

        self.fc = nn.Linear(self.fchidden, self.numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(input_ids=inp_ids,
                            attention_mask=att_mask,
                            token_type_ids=token_ids)
        hidden_states = outputs.hidden_states

        hidden_states = torch.stack([hidden_states[layer_i][:, 0].squeeze()
                                     for layer_i in range(0, self.numlayers)], dim=-1)
        hidden_states = hidden_states.view(-1, self.numlayers, self.embeddim)
        out = self.attention(hidden_states)
        out = self.dropout(out)
        out = self.fc(out)
        return out

    def attention(self, h):
        v = torch.matmul(self.q, h.transpose(-2, -1)).squeeze(1)
        v = F.softmax(v, -1)
        v_temp = torch.matmul(v.unsqueeze(1), h).transpose(-2, -1)
        v = torch.matmul(self.w_h.transpose(1, 0), v_temp).squeeze(2)
        return v

## 5. Training & Evaluation Functions with K-Fold Cross-Validation

Functions for training the model and evaluating performance using 10-Fold CV.

In [10]:
def evaluate(loader, net, device, model_name):
    """ Evaluates a model and returns loss, accuracy

    Arguments:
    loader (DataLoader): dataloader to evaluate
    net (nn.Module): Model to evaluate
    device (torch.device): Device type
    model_name (str): Model name

    """
    net.eval()

    with torch.no_grad():
        loss = 0.0
        total = 0.0
        acc = 0.0
        y_pred = []
        y_true = []
        for input_id, attention_masks, token_ids, labels in loader:
            input_id = input_id.to(device)
            attention_masks = attention_masks.to(device)
            token_ids = token_ids.to(device)
            labels = labels.long().to(device)

            if(model_name == 'base'):
                curloss, output = net(input_id, attention_masks, token_ids, labels)
            else:
                output = net(input_id, attention_masks, token_ids)
                curloss = F.cross_entropy(output, labels, reduction='sum')
            loss += curloss.item()
            preds = torch.argmax(output, 1)
            y_pred.extend(preds.tolist())
            y_true.extend(labels.tolist())
            acc += torch.sum(preds == labels).item()
            total += input_id.size(0)

        F1 = round((f1_score(y_true, y_pred, average='macro')), 2) * 100
        return round((loss / total), 3), round(((acc / total) * 100), 2), F1, y_pred, y_true

In [11]:
def train_single_fold(train_loader, val_loader, model_name, numclasses, numepochs, device):
    """ Trains a model for a single fold

    Arguments:
    train_loader (DataLoader): Training Data Loader
    val_loader (DataLoader): Validation Data Loader
    model_name (str): Name of the model to train
    numclasses (int): Number of classes in the data
    numepochs (int): Number of epochs to train
    device (torch.device): Device type

    Returns:
    best_model: The best trained model
    best_val_acc: Best validation accuracy
    """
    if(model_name == 'lstm'):
        model = Bert_LSTM(numclasses).to(device)
    elif(model_name == 'attention'):
        model = Bert_Attention(numclasses, device).to(device)
    else:
        model = Bert_Base(numclasses).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    model.train()
    valbest = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(1, numepochs+1):
        model.train()
        epoch_loss = 0.0
        num_batches = 0
        
        # Add progress bar for batches
        pbar = tqdm(train_loader, desc=f"  Epoch {epoch}/{numepochs}", leave=False)
        for input_id, attention_masks, token_ids, labels in pbar:
            input_id = input_id.to(device)
            attention_masks = attention_masks.to(device)
            token_ids = token_ids.to(device)
            labels = labels.long().to(device)

            model.zero_grad()

            if(model_name == 'base'):
                loss, _ = model(input_id, attention_masks, token_ids, labels)
            else:
                output = model(input_id, attention_masks, token_ids)
                loss = F.cross_entropy(output, labels)

            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        valloss, valacc, _, _, _ = evaluate(val_loader, model, device, model_name)
        if(valacc > valbest):
            valbest = valacc
            best_model_wts = copy.deepcopy(model.state_dict())

        avg_loss = epoch_loss / num_batches if num_batches > 0 else 0
        print(f"  Epoch {epoch}/{numepochs} - Train Loss: {avg_loss:.4f}, Val Loss: {valloss:.3f}, Val Acc: {valacc:.2f}%")

    model.load_state_dict(best_model_wts)
    return model, valbest

In [12]:
def train_kfold(sentences, aspects, sentiments, bert_tokenizer, model_name,
                numclasses, numepochs, batch_size, maxlen, n_folds, device):
    """ Trains the model using K-Fold Cross-Validation

    Arguments:
    sentences (list): List of sentences
    aspects (list): List of aspects
    sentiments (list): List of sentiment labels
    bert_tokenizer: BERT tokenizer
    model_name (str): Name of the model to train
    numclasses (int): Number of classes in the data
    numepochs (int): Number of epochs to train
    batch_size (int): Batch size
    maxlen (int): Maximum sequence length
    n_folds (int): Number of folds for cross-validation
    device (torch.device): Device type

    Returns:
    fold_results: Dictionary with results for each fold
    best_model: Best model across all folds
    """
    # Convert to numpy arrays for indexing
    sentences = np.array(sentences)
    aspects = np.array(aspects)
    sentiments = np.array(sentiments)
    
    # Initialize StratifiedKFold to maintain class distribution
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    
    fold_results = {
        'fold_acc': [],
        'fold_f1': [],
        'all_preds': [],
        'all_true': []
    }
    
    best_fold_acc = 0.0
    best_model_state = None
    
    print(f"\n{'='*70}")
    print(f"Starting {n_folds}-Fold Cross-Validation for {model_name.upper()} model")
    print(f"{'='*70}")
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(sentences, sentiments), 1):
        print(f"\n{'─'*50}")
        print(f"FOLD {fold}/{n_folds}")
        print(f"Train size: {len(train_idx)}, Validation size: {len(val_idx)}")
        print(f"{'─'*50}")
        
        # Get fold data
        train_sentences = sentences[train_idx].tolist()
        train_aspects = aspects[train_idx].tolist()
        train_sentiments = sentiments[train_idx].tolist()
        
        val_sentences = sentences[val_idx].tolist()
        val_aspects = aspects[val_idx].tolist()
        val_sentiments = sentiments[val_idx].tolist()
        
        # Tokenize
        print("  Tokenizing fold data...")
        train_input_ids, train_attention_masks, train_token_type_ids = tokenize_sentences(
            bert_tokenizer, train_sentences, train_aspects, maxlen)
        train_labels = torch.from_numpy(np.asarray(train_sentiments, 'int32'))
        
        val_input_ids, val_attention_masks, val_token_type_ids = tokenize_sentences(
            bert_tokenizer, val_sentences, val_aspects, maxlen)
        val_labels = torch.from_numpy(np.asarray(val_sentiments, 'int32'))
        
        # Create data loaders
        train_loader = get_loader(train_input_ids, train_attention_masks,
                                  train_token_type_ids, train_labels, batch_size)
        val_loader = get_loader(val_input_ids, val_attention_masks,
                                val_token_type_ids, val_labels, batch_size)
        
        # Train for this fold
        print(f"  Training {model_name.upper()} model...")
        model, best_val_acc = train_single_fold(train_loader, val_loader, model_name,
                                                 numclasses, numepochs, device)
        
        # Evaluate on validation set
        val_loss, val_acc, val_f1, y_pred, y_true = evaluate(val_loader, model, device, model_name)
        
        print(f"\n  ✓ Fold {fold} Results: Accuracy = {val_acc:.2f}%, F1 = {val_f1:.2f}%")
        
        # Store results
        fold_results['fold_acc'].append(val_acc)
        fold_results['fold_f1'].append(val_f1)
        fold_results['all_preds'].extend(y_pred)
        fold_results['all_true'].extend(y_true)
        
        # Track best model
        if val_acc > best_fold_acc:
            best_fold_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())
        
        # Clear GPU memory
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Calculate overall metrics
    avg_acc = np.mean(fold_results['fold_acc'])
    std_acc = np.std(fold_results['fold_acc'])
    avg_f1 = np.mean(fold_results['fold_f1'])
    std_f1 = np.std(fold_results['fold_f1'])
    
    # Overall F1 from all predictions
    overall_f1 = f1_score(fold_results['all_true'], fold_results['all_preds'], average='macro') * 100
    
    print(f"\n{'='*70}")
    print(f"{n_folds}-FOLD CROSS-VALIDATION RESULTS for {model_name.upper()}")
    print(f"{'='*70}")
    print(f"\nPer-Fold Accuracies: {fold_results['fold_acc']}")
    print(f"Per-Fold F1 Scores:  {fold_results['fold_f1']}")
    print(f"\nMean Accuracy: {avg_acc:.2f}% (± {std_acc:.2f}%)")
    print(f"Mean F1 Score: {avg_f1:.2f}% (± {std_f1:.2f}%)")
    print(f"Overall F1 (all predictions): {overall_f1:.2f}%")
    
    # Recreate best model
    if(model_name == 'lstm'):
        best_model = Bert_LSTM(numclasses).to(device)
    elif(model_name == 'attention'):
        best_model = Bert_Attention(numclasses, device).to(device)
    else:
        best_model = Bert_Base(numclasses).to(device)
    
    best_model.load_state_dict(best_model_state)
    best_model.eval()
    
    fold_results['avg_acc'] = avg_acc
    fold_results['std_acc'] = std_acc
    fold_results['avg_f1'] = avg_f1
    fold_results['std_f1'] = std_f1
    fold_results['overall_f1'] = overall_f1
    
    return fold_results, best_model

## 6. Main Pipeline with 10-Fold Cross-Validation

Configure the parameters and run the training pipeline with K-Fold CV.

In [13]:
# Configuration parameters (modify these as needed)
dataset = 'laptop'          # Options: 'laptop', 'restaurant'
maxlen = 80                 # Maximum sentence length for BERT
numclasses = 3              # Number of classes (3 or 4 to include 'conflict')
data_path = '../Data/'      # Path to dataset folder
batch_size = 8              # Batch size
numepochs = 3               # Number of training epochs per fold (reduced for faster training)
n_folds = 5                 # Number of folds (reduced from 10 for faster testing, increase to 10 for final run)

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected - training will be slow!")

print(f"\nConfiguration:")
print(f"  Dataset: {dataset}")
print(f"  Number of folds: {n_folds}")
print(f"  Epochs per fold: {numepochs}")
print(f"  Batch size: {batch_size}")

# Estimate training time
if not torch.cuda.is_available():
    print(f"\n⏱️  Estimated time: ~{n_folds * numepochs * 30} minutes on CPU")
    print("   Consider using only 1 model type first to test")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB

Configuration:
  Dataset: laptop
  Number of folds: 5
  Epochs per fold: 3
  Batch size: 8


In [14]:
# Load dataset (combined train + test for K-Fold CV)
print("Loading combined dataset for K-Fold Cross-Validation...")
all_sentences, all_aspects, all_sentiments = load_semeval_combined(dataset, numclasses, data_path)

# Print class distribution
unique, counts = np.unique(all_sentiments, return_counts=True)
print("\nClass Distribution:")
label_names_map = {0: 'Negative', 1: 'Positive', 2: 'Neutral', 3: 'Conflict'}
for label, count in zip(unique, counts):
    print(f"  {label_names_map[label]}: {count} ({count/len(all_sentiments)*100:.1f}%)")

Loading combined dataset for K-Fold Cross-Validation...
----------------------------------------
Total samples: 2951
Maximum Sentence Length: 78
Maximum Aspect Length: 7
----------------------------------------

Class Distribution:
  Negative: 994 (33.7%)
  Positive: 1328 (45.0%)
  Neutral: 629 (21.3%)


In [15]:
# Load pretrained BERT tokenizer
bert_tokenizer = get_pretrained_tokenizer()

---------------------------------------


In [16]:
# Train models with K-Fold Cross-Validation
# NOTE: Change this list to train fewer models for faster testing
# Options: ['lstm'], ['attention'], ['base'], or ['lstm', 'attention', 'base'] for all
model_names_list = ['lstm']  # Start with just one model to test; change to all 3 for final run

all_results = {}
trained_models = {}

for model_name in model_names_list:
    print(f"\n\n{'#'*80}")
    print(f"# TRAINING {model_name.upper()} MODEL WITH {n_folds}-FOLD CROSS-VALIDATION")
    print(f"{'#'*80}")
    
    results, best_model = train_kfold(
        all_sentences, all_aspects, all_sentiments,
        bert_tokenizer, model_name, numclasses, numepochs,
        batch_size, maxlen, n_folds, DEVICE
    )
    
    all_results[model_name] = results
    trained_models[model_name] = best_model
    
    # Clear GPU memory between models
    torch.cuda.empty_cache() if torch.cuda.is_available() else None



################################################################################
# TRAINING LSTM MODEL WITH 5-FOLD CROSS-VALIDATION
################################################################################

Starting 5-Fold Cross-Validation for LSTM model

──────────────────────────────────────────────────
FOLD 1/5
Train size: 2360, Validation size: 591
──────────────────────────────────────────────────
  Tokenizing fold data...


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Training LSTM model...
BERT Model Loaded


  Epoch 1/3 - Train Loss: 0.7810, Val Loss: 0.671, Val Acc: 71.07%


  Epoch 2/3 - Train Loss: 0.4849, Val Loss: 0.617, Val Acc: 75.63%


...

  Epoch 1/3 - Train Loss: 0.8228, Val Loss: 0.650, Val Acc: 73.90%


  Epoch 2/3 - Train Loss: 0.4928, Val Loss: 0.715, Val Acc: 72.20%


  Epoch 3/3 - Train Loss: 0.3053, Val Loss: 0.966, Val Acc: 63.56%

  ✓ Fold 5 Results: Accuracy = 73.90%, F1 = 71.00%

5-FOLD CROSS-VALIDATION RESULTS for LSTM

Per-Fold Accuracies: [75.63, 77.8, 74.41, 73.22, 73.9]
Per-Fold F1 Scores:  [73.0, 75.0, 71.0, 70.0, 71.0]

Mean Accuracy: 74.99% (± 1.61%)
Mean F1 Score: 72.00% (± 1.79%)
Overall F1 (all predictions): 72.26%


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions"))'), '(Request ID: 045425c9-36cc-4b4a-8f92-71b86e28f46a)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/config.json (Caused by NewConnectionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to establish a new connection: [WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions"))'), '(Request ID: 1e8f603b-2efc-45a6-9f65-9ed2f057fec7)')' thrown while re

BERT Model Loaded


In [17]:
# Print comparison summary
print("\n" + "="*80)
print(f"{n_folds}-FOLD CROSS-VALIDATION COMPARISON SUMMARY")
print("="*80)
print(f"\n{'Model':<12} {'Mean Acc':<15} {'Std Acc':<12} {'Mean F1':<15} {'Std F1':<12} {'Overall F1':<12}")
print("-" * 80)

for name, results in all_results.items():
    print(f"{name.upper():<12} {results['avg_acc']:.2f}%{'':<9} {results['std_acc']:.2f}%{'':<7} "
          f"{results['avg_f1']:.2f}%{'':<9} {results['std_f1']:.2f}%{'':<7} {results['overall_f1']:.2f}%")

# Find best model
best_model_name = max(all_results, key=lambda x: all_results[x]['avg_acc'])
print(f"\n✓ Best performing model: {best_model_name.upper()}")
print(f"  Mean Accuracy: {all_results[best_model_name]['avg_acc']:.2f}% (± {all_results[best_model_name]['std_acc']:.2f}%)")
print(f"  Mean F1 Score: {all_results[best_model_name]['avg_f1']:.2f}% (± {all_results[best_model_name]['std_f1']:.2f}%)")


5-FOLD CROSS-VALIDATION COMPARISON SUMMARY

Model        Mean Acc        Std Acc      Mean F1         Std F1       Overall F1  
--------------------------------------------------------------------------------
LSTM         74.99%          1.61%        72.00%          1.79%        72.26%

✓ Best performing model: LSTM
  Mean Accuracy: 74.99% (± 1.61%)
  Mean F1 Score: 72.00% (± 1.79%)


In [18]:
# Detailed per-fold results
print("\n" + "="*80)
print("DETAILED PER-FOLD RESULTS")
print("="*80)

for model_name, results in all_results.items():
    print(f"\n{model_name.upper()} Model:")
    print(f"  {'Fold':<8} {'Accuracy':<12} {'F1 Score':<12}")
    print(f"  {'-'*32}")
    for fold_idx, (acc, f1) in enumerate(zip(results['fold_acc'], results['fold_f1']), 1):
        print(f"  {fold_idx:<8} {acc:.2f}%{'':<7} {f1:.2f}%")
    print(f"  {'-'*32}")
    print(f"  {'Mean':<8} {results['avg_acc']:.2f}%{'':<7} {results['avg_f1']:.2f}%")
    print(f"  {'Std':<8} {results['std_acc']:.2f}%{'':<7} {results['std_f1']:.2f}%")


DETAILED PER-FOLD RESULTS

LSTM Model:
  Fold     Accuracy     F1 Score    
  --------------------------------
  1        75.63%        73.00%
  2        77.80%        75.00%
  3        74.41%        71.00%
  4        73.22%        70.00%
  5        73.90%        71.00%
  --------------------------------
  Mean     74.99%        72.00%
  Std      1.61%        1.79%


## 7. Save Best Models

Save the best model from each model type.

In [19]:
# Save all best models from K-Fold CV
saved_model_paths = {}

for model_name, model in trained_models.items():
    model_save_path = f'bert_absa_{model_name}_kfold_model.pth'
    torch.save({
        'model_state_dict': model.state_dict(),
        'model_type': model_name,
        'numclasses': numclasses,
        'maxlen': maxlen,
        'n_folds': n_folds,
        'cv_results': all_results[model_name]
    }, model_save_path)
    saved_model_paths[model_name] = model_save_path
    print(f"✓ Saved {model_name.upper()} model to: {model_save_path}")
    print(f"  Cross-validation accuracy: {all_results[model_name]['avg_acc']:.2f}% (± {all_results[model_name]['std_acc']:.2f}%)")

print("\nAll models saved with K-Fold CV results!")

✓ Saved LSTM model to: bert_absa_lstm_kfold_model.pth
  Cross-validation accuracy: 74.99% (± 1.61%)

All models saved with K-Fold CV results!


## 8. Confusion Matrix and Classification Report

In [20]:
# Print classification report for each model (aggregated across all folds)
label_names = ['Negative', 'Positive', 'Neutral']

for model_name, results in all_results.items():
    print(f"\n{'='*60}")
    print(f"CLASSIFICATION REPORT: {model_name.upper()} (All Folds Combined)")
    print(f"{'='*60}")
    print(classification_report(results['all_true'], results['all_preds'], 
                                target_names=label_names[:numclasses]))
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(results['all_true'], results['all_preds'])
    print(cm)


CLASSIFICATION REPORT: LSTM (All Folds Combined)
              precision    recall  f1-score   support

    Negative       0.83      0.64      0.72       994
    Positive       0.85      0.86      0.86      1328
     Neutral       0.52      0.69      0.59       629

    accuracy                           0.75      2951
   macro avg       0.73      0.73      0.72      2951
weighted avg       0.77      0.75      0.75      2951


Confusion Matrix:
[[ 633   98  263]
 [  39 1147  142]
 [  89  107  433]]


## 9. Inference with K-Fold Trained Models

In [21]:
# Function to load a saved model
def load_model(model_path, device):
    """Load a saved model for inference."""
    checkpoint = torch.load(model_path, map_location=device)
    
    model_type = checkpoint['model_type']
    num_classes = checkpoint['numclasses']
    max_len = checkpoint['maxlen']
    
    if model_type == 'lstm':
        model = Bert_LSTM(num_classes).to(device)
    elif model_type == 'attention':
        model = Bert_Attention(num_classes, device).to(device)
    else:
        model = Bert_Base(num_classes).to(device)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    return model, max_len, model_type


# Prediction function
def predict_sentiment(model, bert_tokenizer, sentence, aspect, maxlen, device, model_type='lstm'):
    """Predict sentiment for a sentence-aspect pair."""
    model.eval()
    
    encoded = bert_tokenizer.encode_plus(
        text=sentence,
        text_pair=aspect,
        add_special_tokens=True,
        max_length=maxlen,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors='pt'
    )
    
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    token_type_ids = encoded['token_type_ids'].to(device)
    
    with torch.no_grad():
        if model_type == 'base':
            dummy_label = torch.tensor([0]).to(device)
            _, output = model(input_ids, attention_mask, token_type_ids, dummy_label)
        else:
            output = model(input_ids, attention_mask, token_type_ids)
        
        probabilities = F.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    return predicted.item(), confidence.item()


# Label mapping
label_names_map = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}

In [22]:
# Test examples
test_examples = [
    ("The battery life of this laptop is amazing and lasts all day.", "battery life"),
    ("The screen resolution is terrible and hurts my eyes.", "screen resolution"),
    ("The keyboard is okay, nothing special.", "keyboard"),
    ("I love the fast performance of this machine.", "performance"),
    ("The customer service was unhelpful and rude.", "customer service"),
]

print("="*80)
print("INFERENCE WITH K-FOLD TRAINED MODELS")
print("="*80)

for sentence, aspect in test_examples:
    print(f"\nSentence: \"{sentence}\"")
    print(f"Aspect: \"{aspect}\"")
    print("-" * 70)
    
    for model_name, model in trained_models.items():
        pred_label, confidence = predict_sentiment(model, bert_tokenizer,
                                                    sentence, aspect, maxlen, DEVICE,
                                                    model_type=model_name)
        print(f"  {model_name.upper():<12}: {label_names_map[pred_label]:<10} (confidence: {confidence:.2%})")

INFERENCE WITH K-FOLD TRAINED MODELS

Sentence: "The battery life of this laptop is amazing and lasts all day."
Aspect: "battery life"
----------------------------------------------------------------------
  LSTM        : Positive   (confidence: 98.45%)

Sentence: "The screen resolution is terrible and hurts my eyes."
Aspect: "screen resolution"
----------------------------------------------------------------------
  LSTM        : Negative   (confidence: 93.66%)

Sentence: "The keyboard is okay, nothing special."
Aspect: "keyboard"
----------------------------------------------------------------------
  LSTM        : Positive   (confidence: 97.34%)

Sentence: "I love the fast performance of this machine."
Aspect: "performance"
----------------------------------------------------------------------
  LSTM        : Positive   (confidence: 98.41%)

Sentence: "The customer service was unhelpful and rude."
Aspect: "customer service"
-----------------------------------------------------------